In [1]:
import numpy as np 
import torch, torchvision 
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F 
import torchvision.transforms as transforms 
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt 

In [2]:
torch.manual_seed(20)

In [3]:
if torch.cuda.is_available():
    device = 'cuda'
    print('cuda available')
else:
    device = 'cpu'
    print('only cpu available')

cuda available


In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__() 

        self.rnn = nn.RNN(input_size=input_size,
                          hidden_size=hidden_size,
                         num_layers = num_layers,
                         batch_first=True)

        self.fc = nn.Linear(hidden_size, output_size) 

    def forward(self, x):
        rnn_out, h_n = self.rnn(x) 
        last_output = rnn_out[:, -1, :] 
        output = self.fc(last_output)
        return output 

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.LSTM(embed_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.h2o = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.embed(x)
        _, (h_n, _) = self.rnn(x)
        return self.h2o(h_n[-1])  # raw logits; use CrossEntropyLoss